# HuggingFace Backup - CaML

Backs up models from `CompassioninMachineLearning` to `Backup-CaML`.

**Colab free tier:** ~100GB disk space (enough for models up to ~45GB)

In [ ]:
# Check disk space
!df -h /
!nvidia-smi || echo 'No GPU (fine for backup)'

In [ ]:
# Install dependencies
!pip install -q huggingface_hub tqdm

In [ ]:
# Authenticate with HuggingFace
# You'll need a token with write access to Backup-CaML
from huggingface_hub import login
login()  # Will prompt for token

In [ ]:
# Verify access
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()
print(f"Logged in as: {user['name']}")

orgs = {org['name']: org.get('roleInOrg', 'unknown') for org in user.get('orgs', [])}
print(f"CompassioninMachineLearning: {orgs.get('CompassioninMachineLearning', 'NOT FOUND')}")
print(f"Backup-CaML: {orgs.get('Backup-CaML', 'NOT FOUND')}")

In [ ]:
# List models that need backup (not yet in Backup-CaML)
from huggingface_hub import HfApi

api = HfApi()
SOURCE_ORG = "CompassioninMachineLearning"
BACKUP_ORG = "Backup-CaML"

# Get all source models
source_models = list(api.list_models(author=SOURCE_ORG))
print(f"Source org has {len(source_models)} models")

# Get all backup models
backup_models = list(api.list_models(author=BACKUP_ORG))
backup_names = {m.id.split('/')[-1] for m in backup_models}
print(f"Backup org has {len(backup_models)} models")

# Find models needing backup
needs_backup = []
for m in source_models:
    name = m.id.split('/')[-1]
    if name not in backup_names:
        # Get size
        try:
            info = api.model_info(m.id, files_metadata=True)
            size = sum(f.size for f in info.siblings if f.size) if info.siblings else 0
            needs_backup.append((name, size / (1024**3)))
        except:
            needs_backup.append((name, 0))

# Sort largest first (Colab has ~100GB, use it for big models)
needs_backup.sort(key=lambda x: -x[1])
print(f"\nModels needing backup: {len(needs_backup)}")
print("\nBy size (largest first):")
for name, gb in needs_backup[:15]:
    print(f"  {gb:6.2f} GB  {name}")

In [ ]:
# Backup function
import tempfile
import shutil
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
import os

def get_repo_files(api, repo_id):
    """Get dict of {filename: size} for a repo."""
    try:
        info = api.model_info(repo_id, files_metadata=True)
        if info.siblings:
            return {f.rfilename: f.size for f in info.siblings}
    except:
        pass
    return None

def backup_model(model_name, source_org="CompassioninMachineLearning", backup_org="Backup-CaML"):
    """Backup a single model."""
    api = HfApi()
    source_id = f"{source_org}/{model_name}"
    backup_id = f"{backup_org}/{model_name}"
    
    # Get source file info
    source_files = get_repo_files(api, source_id)
    if source_files is None:
        print(f"❌ {model_name}: Source not found")
        return False
    
    size_gb = sum(source_files.values()) / (1024**3)
    
    # Check if backup exists AND matches source
    backup_files = get_repo_files(api, backup_id)
    if backup_files is not None:
        if backup_files == source_files:
            print(f"⏭️  {model_name}: Already backed up correctly ({size_gb:.2f} GB)")
            return True
        else:
            # Backup exists but doesn't match - need to re-upload
            missing = set(source_files.keys()) - set(backup_files.keys())
            extra = set(backup_files.keys()) - set(source_files.keys())
            size_diff = [f for f in source_files if f in backup_files and source_files[f] != backup_files[f]]
            print(f"⚠️  {model_name}: Backup incomplete/mismatched, re-uploading...")
            if missing:
                print(f"   Missing files: {len(missing)}")
            if size_diff:
                print(f"   Size mismatches: {len(size_diff)}")
    
    print(f"📦 {model_name}: {size_gb:.2f} GB")
    
    # Check disk space
    free_gb = shutil.disk_usage('/').free / (1024**3)
    if size_gb > free_gb - 5:  # Keep 5GB buffer
        print(f"❌ Not enough disk space (need {size_gb:.1f}GB, have {free_gb:.1f}GB)")
        return False
    
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            local_path = Path(tmpdir) / model_name
            
            # Download
            print(f"   ⬇️  Downloading...")
            snapshot_download(
                repo_id=source_id,
                local_dir=str(local_path),
                repo_type="model",
            )
            
            # Show download complete with size
            downloaded_size = sum(f.stat().st_size for f in local_path.rglob('*') if f.is_file()) / (1024**3)
            print(f"   ⬇️  Downloaded {downloaded_size:.2f} GB")
            
            # Create backup repo (public)
            print(f"   📝 Creating backup repo...")
            api.create_repo(
                repo_id=backup_id,
                repo_type="model",
                exist_ok=True,
                private=False,
            )
            
            # Upload with progress
            print(f"   ⬆️  Uploading {downloaded_size:.2f} GB...")
            api.upload_folder(
                folder_path=str(local_path),
                repo_id=backup_id,
                repo_type="model",
            )
            print(f"   ⬆️  Upload complete")
            
            # Verify upload matches source
            new_backup_files = get_repo_files(api, backup_id)
            if new_backup_files == source_files:
                print(f"✅ {model_name}: Backup complete and verified")
                return True
            else:
                print(f"⚠️  {model_name}: Upload completed but verification failed!")
                return False
            
    except Exception as e:
        print(f"❌ {model_name}: Failed - {e}")
        return False

In [ ]:
# Backup a single model (test)
# Uncomment and change the model name to test

# backup_model("some_model_name")

In [ ]:
# Backup all remaining models (LARGEST first - use Colab's 100GB for big ones)
# This will take a while!

from tqdm import tqdm
import shutil

# Get fresh list of what needs backup
source_models = list(api.list_models(author="CompassioninMachineLearning"))
backup_models = list(api.list_models(author="Backup-CaML"))
backup_names = {m.id.split('/')[-1] for m in backup_models}

to_backup = []
for m in source_models:
    name = m.id.split('/')[-1]
    if name not in backup_names:
        try:
            info = api.model_info(m.id, files_metadata=True)
            size = sum(f.size for f in info.siblings if f.size) if info.siblings else 0
            to_backup.append((name, size))
        except:
            to_backup.append((name, 0))

# Sort by size (LARGEST first)
to_backup.sort(key=lambda x: -x[1])

print(f"Backing up {len(to_backup)} models (largest first)...\n")
print("Top 5:")
for name, size in to_backup[:5]:
    print(f"  {size/(1024**3):.1f} GB  {name}")
print()

# Show initial disk space
free_gb = shutil.disk_usage('/').free / (1024**3)
print(f"Disk free: {free_gb:.1f} GB\n")

success = 0
failed = 0

for name, size in tqdm(to_backup):
    if backup_model(name):
        success += 1
    else:
        failed += 1
    
    # Clean up any leftover temp files
    !rm -rf /tmp/tmp* 2>/dev/null
    
    # Show disk space after cleanup
    free_gb = shutil.disk_usage('/').free / (1024**3)
    print(f"   Disk free: {free_gb:.1f} GB\n")

print(f"\n=== DONE ===")
print(f"Success: {success}")
print(f"Failed: {failed}")

In [ ]:
# Verify backups match source
from huggingface_hub import HfApi

api = HfApi()
backups = list(api.list_models(author="Backup-CaML"))

print(f"Verifying {len(backups)} backups...\n")

correct = 0
mismatch = []

for backup in backups:
    name = backup.id.split("/")[-1]
    source_id = f"CompassioninMachineLearning/{name}"
    
    try:
        src_info = api.model_info(source_id, files_metadata=True)
        bak_info = api.model_info(backup.id, files_metadata=True)
        
        src_files = {f.rfilename: f.size for f in src_info.siblings} if src_info.siblings else {}
        bak_files = {f.rfilename: f.size for f in bak_info.siblings} if bak_info.siblings else {}
        
        if src_files == bak_files:
            correct += 1
        else:
            mismatch.append(name)
            print(f"MISMATCH: {name}")
    except Exception as e:
        print(f"ERROR: {name} - {e}")

print(f"\nCorrect: {correct}")
print(f"Mismatch: {len(mismatch)}")
if mismatch:
    print(f"Mismatched repos: {mismatch}")

In [ ]:
# Check disk space
!df -h /